# Métricas de Avaliação



In [1]:
# Importando bibliotecas
import pandas as pd
#Split dos dados
from sklearn.model_selection import train_test_split
#Modelos
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

#Métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score)

#MLflow
from pathlib import Path
import mlflow
import mlflow.sklearn
# Entendendo o caminho do projeto para salvar o banco de dados do MLflow
ROOT = Path.cwd().resolve().parent
# Usa banco SQLite (mais estável)
TRACKING_URI = f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)


In [2]:
# Carregando os dados
dados = pd.read_csv("../data/processed/dados_limpos.csv")

In [3]:
# Separando as features e o target
X = dados.drop("Churn", axis=1)
y = dados["Churn"]


In [4]:
# Dividindo os dados em treino e teste
seed = 2711
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed)

In [5]:
# Treinando as métricas

def calcular_metricas(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }


In [6]:
#Função para calcular o custo de negócio
def custo_negocio(y_true, y_pred):
    """
    Calcula o impacto financeiro das previsões do modelo.
    Dicionário:
    * FP = Falso Positivo
    * FN = Falso Negativo
    * 100 = custo de retenção por cliente
    * 840 = perda média por cliente que cancela
    Regras:
    - FP (Falso Positivo): cliente não ia sair, mas recebeu oferta → custo menor (R$ 100) -- desconto
    - FN (Falso Negativo): cliente ia sair e não foi identificado → custo alto (R$ 840) -- Cliente perdido

    Objetivo:
    Minimizar esse custo, principalmente evitando perder clientes (FN).
    """

    # Clientes que não iam sair, mas o modelo disse que iam
    falsos_positivos = ((y_pred == 1) & (y_true == 0)).sum()

    # Clientes que iam sair, mas o modelo não identificou
    falsos_negativos = ((y_pred == 0) & (y_true == 1)).sum()

    # Cálculo do custo total
    custo_total = (falsos_positivos * 100) + (falsos_negativos * 840)

    return custo_total

In [7]:
#Treinando os modelos
modelos = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "LogisticRegression": LogisticRegression(max_iter=2000),
}

resultados = []

# Definindo o nome do experimento no MLflow
mlflow.set_experiment("churn_baseline")

for nome, modelo in modelos.items():

    with mlflow.start_run(run_name=nome):
        # Tratamento especial para Logistic Regression
        if nome == "LogisticRegression":

            # Normaliza os dados (necessário para o modelo funcionar bem)
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            modelo.fit(X_train_scaled, y_train)
            y_pred = modelo.predict(X_test_scaled)

        else:
            # Outros modelos não precisam de normalização
            modelo.fit(X_train, y_train)
            y_pred = modelo.predict(X_test)

        # Métricas
        metricas = calcular_metricas(y_test, y_pred)
        custo = custo_negocio(y_test, y_pred)
        metricas["custo_negocio"] = custo
        metricas["modelo"] = nome

        resultados.append(metricas)

        # -------------------------
        # LOG NO MLFLOW
        # -------------------------

        # Parâmetros do modelo
        mlflow.log_param("modelo", nome)

        # Métricas (somente números)
        for k, v in metricas.items():
            if isinstance(v, (int, float)):
                mlflow.log_metric(k, v)

        # Métrica de negócio
        mlflow.log_metric("custo_negocio", custo)

        # Dataset (versão simples)
        mlflow.log_param("dataset", "dados_limpos_v1")

        # Salvar modelo
        mlflow.sklearn.log_model(
        sk_model=modelo,
        name="model",
        registered_model_name=f"churn_{nome.lower()}"
        )



2026/03/27 21:24:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'churn_dummy' already exists. Creating a new version of this model...
Created version '2' of model 'churn_dummy'.
2026/03/27 21:24:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'churn_decisiontree' already exists. Creating a ne

In [8]:
#Criando tabela de comparação de resultados

df_resultados = pd.DataFrame(resultados)

#Hearder
df_resultados = df_resultados[
    ["modelo", "accuracy", "precision", "recall", "f1", "custo_negocio"]
]

#Ordenar por acurácia
df_resultados = df_resultados.sort_values(by="accuracy", ascending=False)

#Formatar o custo de negócio
df_resultados["custo_negocio"] = df_resultados["custo_negocio"].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

In [9]:
#Exibindo os resultados

# A coluna custo_negocio representa o impacto financeiro das decisões do modelo.
#
# Os valores são altos porque consideramos o valor total de um cliente ao longo do tempo,
# e não apenas o valor mensal.
#
# - Falso Negativo (FN): cliente que iria cancelar e não foi identificado → perda alta (~R$ 840)
# - Falso Positivo (FP): cliente que não iria cancelar, mas recebeu oferta → custo menor (~R$ 100)
#
# Como o custo de perder um cliente é muito maior do que o custo de tentar retê-lo,
# o valor final do custo_negocio tende a ser elevado.
#
# O objetivo do modelo é reduzir esse custo, principalmente diminuindo os falsos negativos.

df_resultados

,modelo,accuracy,precision,recall,f1,custo_negocio
4,LogisticRegression,0.806246,0.699346,0.541772,0.610556,"R$ 161.240,00"
2,RandomForest,0.784244,0.648208,0.503797,0.566952,"R$ 175.440,00"
3,KNN,0.771469,0.627178,0.455696,0.527859,"R$ 191.300,00"
1,DecisionTree,0.730305,0.520325,0.486076,0.502618,"R$ 188.220,00"
0,Dummy,0.719659,0.000000,0.000000,0.000000,"R$ 331.800,00"
